# 🛡️ Trust Before Answer: Ethical AI Guardrail Agent

## Project Summary

This project implements a **"Trust Before Answer"** system — an AI agent that incorporates ethical guardrails to **prevent hallucination** and ensure **knowledge sufficiency** before generating responses. The system uses a **Retrieval-Augmented Generation (RAG)** pipeline combined with a guardrail mechanism to guarantee that every answer is grounded in verified knowledge.

**Knowledge Source**: [help.webex.com](https://help.webex.com) — Cisco Webex documentation covering meetings, messaging, security, and more.

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Hallucination Prevention** | The system refuses to answer when it lacks sufficient context, preventing the LLM from fabricating information |
| **Knowledge Sufficiency Check** | Before generating any answer, the system verifies that relevant context exists in its Webex knowledge base |
| **Ethical AI Guardrail** | A prompt-engineered constraint that forces the model to only answer using provided evidence |
| **RAG Pipeline** | Retrieval-Augmented Generation ensures answers are grounded in actual Webex documentation |

### Architecture Flow

```
User Query → [Retrieval from Webex KB] → [Knowledge Sufficiency Check] → [Guardrail LLM Generation] → Response
                                              ↓ (if no context found)
                                        REFUSAL (safe denial)
```

### Models & Technologies Used

- **LLM**: Google Flan-T5-Base (runs locally, **NO API key required**)
- **Knowledge Source**: help.webex.com (Cisco Webex documentation)
- **Framework**: LangChain (prompt templates, output parsers, chain composition)
- **UI**: Gradio (interactive web interface)
- **Paradigm**: Retrieval-Augmented Generation with ethical constraints

---

## Block 1: Package Installation

This block installs all the required Python packages:

- **`transformers`**: Hugging Face library for loading pre-trained models (Flan-T5)
- **`torch`**: PyTorch deep learning framework (model inference backend)
- **`langchain-huggingface`**: LangChain integration with Hugging Face models
- **`langchain-core`**: Core LangChain components (prompts, parsers, chains)
- **`gradio`**: Web UI framework for building interactive ML demos

**No API key is needed** — all models run locally on your machine.

In [1]:
# Block 1: Install required packages
# All models run locally — NO API key needed!

import subprocess
import sys

packages = [
    "transformers",
    "torch",
    "langchain-huggingface",
    "langchain-core",
    "langchain-text-splitters",
    "PyPDF2",
    "gradio>=4.44.0",
    "gradio_client>=1.3.0"
]

for pkg in packages:
    print(f"Installing {pkg}...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )

print("\n✅ All packages installed successfully!")
print("   No API key required — models run locally.")

Installing transformers...
Installing torch...
Installing langchain-huggingface...
Installing langchain-core...
Installing langchain-text-splitters...
Installing PyPDF2...
Installing gradio>=4.44.0...
Installing gradio_client>=1.3.0...

✅ All packages installed successfully!
   No API key required — models run locally.


## Block 2: Import Libraries

This block imports all necessary modules:

| Library | Purpose |
|---------|--------|
| `os` | File path operations |
| `warnings` | Suppress noisy warnings from transformers |
| `gradio` | Building the interactive web UI |
| `transformers` | Loading and running the Flan-T5 model locally |
| `HuggingFacePipeline` | LangChain wrapper to use HF models in chains |
| `PromptTemplate` | Structured prompt engineering with variables |
| `StrOutputParser` | Parses LLM output into plain strings |

**No API key setup needed!** Everything runs on your local machine.

In [3]:
# Block 2: Import libraries
# No API key configuration needed!

import os
import warnings
warnings.filterwarnings("ignore")

import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


## Block 3: Initialize the Local Language Model

This block downloads and loads the **Google Flan-T5-Base** model locally:

- **`google/flan-t5-base`**: A 250M parameter instruction-tuned model that runs on any machine
- **Runs entirely offline** after first download (model is cached)
- **No API key, no internet needed** after initial download
- **`do_sample=False`**: Deterministic output for consistent guardrail behavior

### Why Flan-T5?

| Feature | Benefit |
|---------|--------|
| Free & open-source | No costs, no API keys |
| Instruction-tuned | Follows prompt instructions well |
| Small (250M params) | Runs on any laptop without GPU |
| Fast inference | Quick response times |

The first run will download the model (~990MB). Subsequent runs use the cached version.

In [4]:
# Block 3: Initialize the local LLM (Flan-T5)
# Downloads the model on first run, then uses cache
# NO API KEY NEEDED!

MODEL_NAME = "google/flan-t5-base"

print(f"⏳ Loading model: {MODEL_NAME}")
print("   (First run downloads ~990MB, subsequent runs use cache)")

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Create a text2text-generation pipeline
pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,  # Deterministic output (like temperature=0)
)

# Wrap in LangChain for chain composition
llm = HuggingFacePipeline(pipeline=pipe)

print(f"\n✅ Model loaded successfully: {MODEL_NAME}")
print("   - Runs locally (no internet needed after download)")
print("   - No API key required")
print("   - Deterministic mode: ON (do_sample=False)")

⏳ Loading model: google/flan-t5-base
   (First run downloads ~990MB, subsequent runs use cache)


Device set to use mps:0



✅ Model loaded successfully: google/flan-t5-base
   - Runs locally (no internet needed after download)
   - No API key required
   - Deterministic mode: ON (do_sample=False)


## Block 4: Knowledge Retrieval Function

This block implements the **retrieval** component of the RAG pipeline. The `retrieve_context()` function simulates searching a knowledge base for information relevant to the user's query.

### How it works:

1. A **mock knowledge base** (dictionary) stores verified facts sourced from [help.webex.com](https://help.webex.com)
2. The function performs **keyword matching** — if the user's query (lowercased) contains a key from the knowledge base, the corresponding fact is returned
3. If **no match is found**, the function returns `None` — this signals to the pipeline that there is insufficient knowledge to answer safely

### Knowledge Base Entries (sourced from help.webex.com):

| Keyword | Topic |
|---------|-------|
| `meeting` | Scheduling and joining Webex meetings |
| `messaging` / `space` | Webex messaging, spaces, and collaboration features |
| `share` / `content` | Sharing content during meetings |
| `noise` / `background` | Noise removal and virtual backgrounds |
| `security` / `encryption` | End-to-end encryption and security features |
| `api` / `bot` | Webex developer APIs and bot accounts |

### Production Note:
In a real deployment, this function would be replaced with a **vector store search** (e.g., FAISS, Pinecone, ChromaDB) that performs semantic similarity matching against embedded documents from help.webex.com.

In [5]:
# Block 4: Knowledge Retrieval Function
# Simulates retrieval from a verified knowledge base (sourced from help.webex.com)

def retrieve_context(query):
    """
    Simulates retrieval from a Webex knowledge base.
    
    Content sourced from help.webex.com articles.
    In a production system, replace this with actual vector store search
    (e.g., FAISS, Pinecone, ChromaDB) for semantic similarity matching.
    
    Args:
        query (str): The user's question/input
    
    Returns:
        str or None: Retrieved context if found, None if no relevant info exists
    """
    # Knowledge base with verified facts from help.webex.com
    knowledge_base = {
        "meeting": (
            "In Webex App, you can schedule meetings directly from a space or "
            "from the Meetings tab. You can join meetings with a single tap across "
            "any device — desktop, phone, or car — using the Move to Mobile QR code "
            "feature and Apple CarPlay integration. Webex supports real-time "
            "translation to 100+ languages, closed captioning, gesture recognition "
            "for in-meeting reactions, and Webex Assistant for voice commands, "
            "transcription, and automatic note-taking. Meetings can scale from "
            "small team calls up to webinars with 10,000 attendees."
        ),
        "messaging": (
            "Webex Messaging keeps work flowing between meetings with asynchronous "
            "collaboration. You can create spaces organized by workstreams, "
            "edit, @mention, forward, flag, pin, and thread messages. Spaces support "
            "file sharing up to 100MB per attachment with formats including Word, Excel, "
            "PowerPoint, PDF, JPEG, PNG, GIF, and BMP. You can co-edit documents, "
            "access meeting artifacts, and collaborate with anyone outside your "
            "organization by adding their email to a space. Webex integrates with "
            "100+ third-party apps including Google, Microsoft, Salesforce, and Slack."
        ),
        "space": (
            "Webex Spaces allow you to organize team communication by workstreams. "
            "You can create spaces for specific projects or topics, add internal "
            "and external participants by email, share files, co-edit documents, "
            "and schedule meetings directly within a space. Spaces support threading, "
            "pinning important messages, advanced search with filters, end-to-end "
            "encryption, and two-way whiteboarding for visual collaboration."
        ),
        "share": (
            "In Webex meetings, you can share your entire screen, a specific "
            "application window, or a file directly. Webex supports immersive share "
            "for presentations, allowing your video to appear integrated with your "
            "content. Supported file attachment types include Microsoft Word (doc, docx), "
            "Excel (xls, xlsx), PowerPoint (ppt, pptx), PDF, JPEG, BMP, GIF, and PNG. "
            "Files can be shared via local upload (multipart/form-data) or remote URL. "
            "The maximum file attachment size is 100MB per file."
        ),
        "content": (
            "In Webex meetings, you can share your entire screen, a specific "
            "application window, or a file directly. Webex supports immersive share "
            "for presentations, allowing your video to appear integrated with your "
            "content. Supported file attachment types include Microsoft Word (doc, docx), "
            "Excel (xls, xlsx), PowerPoint (ppt, pptx), PDF, JPEG, BMP, GIF, and PNG. "
            "Files can be shared via local upload (multipart/form-data) or remote URL. "
            "The maximum file attachment size is 100MB per file."
        ),
        "noise": (
            "Webex includes built-in noise removal technology that filters out "
            "background sounds such as keyboard typing, dog barking, or construction "
            "noise, letting you meet with confidence from any location. Additionally, "
            "voice optimization ensures your speech is clear and natural-sounding "
            "regardless of your environment."
        ),
        "background": (
            "Webex App allows you to use a virtual or blurred background in calls "
            "and meetings. You can choose from preset backgrounds, upload your own "
            "custom image, or blur your surroundings for privacy. Webex also features "
            "built-in noise removal technology and people-focused views that frame "
            "participants optimally regardless of their position in the room."
        ),
        "security": (
            "Webex provides end-to-end encryption for messages, files, and meetings. "
            "Organizations can enable anti-malware scanning of files to protect users "
            "from malicious content. Files under evaluation are quarantined and scanned; "
            "infected files become permanently unavailable for download (410 Gone). "
            "Webex requires Server Name Indication (SNI) for TLS/SSL connections and "
            "uses enterprise-grade security managed through Cisco Control Hub."
        ),
        "encryption": (
            "Webex provides end-to-end encryption for messages, files, and meetings. "
            "Organizations can enable anti-malware scanning of files to protect users "
            "from malicious content. Files under evaluation are quarantined and scanned; "
            "infected files become permanently unavailable for download (410 Gone). "
            "Webex requires Server Name Indication (SNI) for TLS/SSL connections and "
            "uses enterprise-grade security managed through Cisco Control Hub."
        ),
        "api": (
            "The Webex REST API supports pagination using RFC5988 Web Linking standard "
            "with Link headers containing rel='next' for navigating pages. The API "
            "enforces rate limits of approximately 300 requests per minute, returning "
            "429 Too Many Requests with a Retry-After header when exceeded. Message "
            "attachments are limited to 100MB each and can be sent via local file upload "
            "(multipart/form-data) or remote URL. Standard HTTP response codes are used: "
            "200 OK, 201 Created, 401 Unauthorized, 429 Too Many Requests, etc."
        ),
        "bot": (
            "Webex bot accounts have less restrictive rate limits than end-user "
            "accounts, making them suitable for automation. Bots can send messages with "
            "attachments, use Markdown formatting (bold, italic, lists, code blocks, "
            "@mentions), and interact with rooms and people via the REST API. Due to "
            "content ownership rules, there are known issues when bots create large "
            "numbers of messaging spaces. For large workloads, partitioning across "
            "separate bot accounts is recommended."
        ),
        "assistant": (
            "Webex Assistant is the first digital in-meeting assistant for the "
            "enterprise. It supports voice commands during meetings, provides real-time "
            "and recorded transcripts, closed captioning, and automatic highlights and "
            "notes. It handles time-consuming tasks like note taking and action items, "
            "helping teams stay focused on collaboration rather than documentation."
        ),
    }
    
    # Keyword matching — check if any knowledge base key appears in the query
    # In production: use embedding similarity search
    query_lower = query.lower()
    for key, value in knowledge_base.items():
        if key in query_lower:
            return value
    
    return None  # Returns None if no relevant info is found


# --- Test the retrieval function ---
print("Testing retrieve_context() with Webex knowledge base:")
print("-" * 60)
print(f"  Query: 'How do I schedule a meeting in Webex?'")
print(f"  Result: {retrieve_context('How do I schedule a meeting in Webex?')}")
print()
print(f"  Query: 'How do I share content in a meeting?'")
print(f"  Result: {retrieve_context('How do I share content in a meeting?')}")
print()
print(f"  Query: 'Tell me about Webex security and encryption'")
print(f"  Result: {retrieve_context('Tell me about Webex security and encryption')}")
print()
print(f"  Query: 'What is the best pizza place?' (out of scope)")
print(f"  Result: {retrieve_context('What is the best pizza place?')}")
print()
print("✅ Retrieval function working correctly (source: help.webex.com)")

Testing retrieve_context() with Webex knowledge base:
------------------------------------------------------------
  Query: 'How do I schedule a meeting in Webex?'
  Result: In Webex App, you can schedule meetings directly from a space or from the Meetings tab. You can join meetings with a single tap across any device — desktop, phone, or car — using the Move to Mobile QR code feature and Apple CarPlay integration. Webex supports real-time translation to 100+ languages, closed captioning, gesture recognition for in-meeting reactions, and Webex Assistant for voice commands, transcription, and automatic note-taking. Meetings can scale from small team calls up to webinars with 10,000 attendees.

  Query: 'How do I share content in a meeting?'
  Result: In Webex App, you can schedule meetings directly from a space or from the Meetings tab. You can join meetings with a single tap across any device — desktop, phone, or car — using the Move to Mobile QR code feature and Apple CarPlay integrat

## Block 5: Guardrail Prompt Template & Chain

This is the **core innovation** of the project — the **Ethical AI Guardrail**. This block defines a carefully engineered prompt that constrains the LLM's behavior:

### Prompt Design Principles:

1. **Role Assignment**: The LLM is instructed to only answer from the given context
2. **Context Grounding**: The model must analyze the provided context before answering
3. **Explicit Refusal Instruction**: If context is empty or insufficient, the model MUST output a refusal message
4. **Answer Constraint**: Answers must be based **ONLY** on the provided context — no external knowledge allowed

### LangChain Chain Composition:

The chain uses **LCEL (LangChain Expression Language)** pipe syntax:

```
guardrail_prompt → llm (Flan-T5 local) → StrOutputParser()
```

This creates a composable pipeline:
1. **`guardrail_prompt`**: Formats the template with `{context}` and `{question}` variables
2. **`llm`**: Sends the formatted prompt to the local Flan-T5 model
3. **`StrOutputParser()`**: Extracts the text response as a plain string

### Why This Prevents Hallucination:

The prompt explicitly tells the model to:
- Only use provided context
- Refuse when knowledge is insufficient
- Never fabricate information beyond what's given

In [6]:
# Block 5: Guardrail Prompt Template & LangChain Chain
# This is the ethical constraint that prevents hallucination

# Define the prompt structure for the guardrail
guardrail_template = """Answer the question based only on the following context. 
If the context does not contain enough information, say "INSUFFICIENT_KNOWLEDGE: I cannot answer this."

Context: {context}

Question: {question}

Answer:"""

# Create the prompt template with variable placeholders
guardrail_prompt = PromptTemplate(
    template=guardrail_template,
    input_variables=["context", "question"]
)

# Compose the LangChain chain using LCEL (pipe syntax)
# Flow: Prompt → Local LLM (Flan-T5) → String Output
guardrail_chain = guardrail_prompt | llm | StrOutputParser()

print("✅ Guardrail chain created successfully")
print("   Pipeline: PromptTemplate → Flan-T5 (local) → StrOutputParser")
print("   Variables: {context}, {question}")
print("   No API calls — everything runs on your machine!")

✅ Guardrail chain created successfully
   Pipeline: PromptTemplate → Flan-T5 (local) → StrOutputParser
   Variables: {context}, {question}
   No API calls — everything runs on your machine!


## Block 5B: Self-Verification Chain (Answer Validation)

This block implements the **Self-Verifying AI Agent** technique inspired by [LangChain's RAG patterns](https://python.langchain.com/docs/use_cases/question_answering/). The agent **validates its own generated answer** against the retrieved evidence before presenting it to the user.

### How Self-Verification Works:

```
Generated Answer + Original Context → [Verification LLM] → SUPPORTED / NOT_SUPPORTED
```

### Verification Logic:

1. The guardrail chain generates an initial answer from the context
2. The **verification chain** receives:
   - The original retrieved context (evidence)
   - The generated answer (claim)
3. The verification chain checks: *"Is this answer fully supported by the evidence?"*
4. If the answer is **NOT supported**, the system flags it and refuses to present it

### Why This Matters:

| Scenario | Without Verification | With Verification |
|----------|---------------------|-------------------|
| LLM extrapolates beyond context | Answer presented as fact ❌ | Caught and refused ✅ |
| LLM partially hallucinates | Mixed truth/fabrication shown ❌ | Flagged for user ✅ |
| Answer is well-grounded | Answer shown ✅ | Answer confirmed & shown ✅ |

### Three-Layer Protection (Updated Architecture):

```
Layer 1: HARD GUARDRAIL    → Code-level check (context == None → refuse)
Layer 2: SOFT GUARDRAIL    → Prompt instructs "only use context"
Layer 3: SELF-VERIFICATION → LLM validates its own answer against evidence (NEW!)
```

This technique transforms the system from a single-pass generator into a **self-auditing agent** that cross-checks its own outputs.

In [7]:
# Block 5B: Self-Verification Chain
# The agent validates its own answer against the retrieved evidence

# Verification prompt — checks if the generated answer is supported by evidence
verification_template = """You are a verification agent. Your job is to determine if the given answer is fully supported by the provided evidence.

Evidence: {evidence}

Answer to verify: {answer}

Question: Is the above answer fully supported by the evidence? Respond with only "SUPPORTED" if the answer is faithful to the evidence, or "NOT_SUPPORTED" if the answer contains claims not found in the evidence.

Verdict:"""

# Create the verification prompt template
verification_prompt = PromptTemplate(
    template=verification_template,
    input_variables=["evidence", "answer"]
)

# Compose the verification chain
# Flow: Verification Prompt → Local LLM (Flan-T5) → String Output
verification_chain = verification_prompt | llm | StrOutputParser()

print("✅ Self-Verification chain created successfully")
print("   Pipeline: VerificationPrompt → Flan-T5 (local) → StrOutputParser")
print("   Variables: {evidence}, {answer}")
print("   Purpose: Validates generated answers against retrieved evidence")
print("   Technique: Self-Verifying AI Agent (LangChain RAG pattern)")


✅ Self-Verification chain created successfully
   Pipeline: VerificationPrompt → Flan-T5 (local) → StrOutputParser
   Variables: {evidence}, {answer}
   Purpose: Validates generated answers against retrieved evidence
   Technique: Self-Verifying AI Agent (LangChain RAG pattern)


## Block 6: Self-Verifying Agent Pipeline (Core Logic)

This block implements the **main agent pipeline** — now enhanced with the **Self-Verifying AI Agent** technique from LangChain. The agent orchestrates retrieval, generation, and **internal answer verification** before presenting any response.

### Pipeline Steps:

```
Step 1: RETRIEVE       → Search knowledge base for relevant context
Step 2: HARD GUARDRAIL → Check if context is sufficient (None check)
Step 3: GENERATE       → Invoke the guardrail-constrained LLM chain
Step 4: SELF-VERIFY    → Validate generated answer against evidence (NEW!)
Step 5: PRESENT        → Only show verified answer, or flag unverified ones
```

### Decision Logic:

```
if context == None:
    → REFUSE (safe denial, no LLM call needed)
else:
    answer = generate(context, question)
    verification = verify(evidence=context, answer=answer)
    if verification == "SUPPORTED":
        → PRESENT verified answer ✅
    else:
        → FLAG as unverified ⚠️
```

### Three Layers of Protection:

1. **Layer 1 (Code-level)**: If `retrieve_context()` returns `None`, immediate refusal — hard guardrail
2. **Layer 2 (Prompt-level)**: LLM prompt instructs "only use context" — soft guardrail
3. **Layer 3 (Self-Verification)**: A second LLM pass validates the answer against evidence — verification guardrail (NEW!)

This transforms the system into a **self-auditing agent** that cross-checks its own outputs before presenting them.

In [8]:
# Block 6: Self-Verifying Agent Pipeline — Trust Before Answer with internal verification
# Now supports both knowledge base queries AND uploaded document queries

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter for document chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # chunk size in characters
    chunk_overlap=100,    # overlap between chunks for context continuity
    add_start_index=True, # track position in original document
)


def chunk_document(file_path):
    """
    Reads an uploaded document and splits it into chunks.
    
    Supports: .txt, .md, .pdf (text extraction), .csv
    
    Args:
        file_path (str): Path to the uploaded file
    
    Returns:
        list[str]: List of text chunks from the document
    """
    if file_path is None:
        return []
    
    file_ext = os.path.splitext(file_path)[1].lower()
    
    # Read file content based on type
    if file_ext == ".pdf":
        try:
            import PyPDF2
            text = ""
            with open(file_path, "rb") as f:
                reader = PyPDF2.PdfReader(f)
                for page in reader.pages:
                    text += page.extract_text() or ""
        except ImportError:
            # Fallback if PyPDF2 not installed
            with open(file_path, "r", errors="ignore") as f:
                text = f.read()
    else:
        # .txt, .md, .csv, and other text files
        with open(file_path, "r", errors="ignore") as f:
            text = f.read()
    
    if not text.strip():
        return []
    
    # Split into chunks using LangChain's text splitter
    chunks = text_splitter.split_text(text)
    return chunks


def retrieve_from_chunks(query, chunks):
    """
    Simple keyword-based retrieval from document chunks.
    Returns the most relevant chunks based on keyword overlap.
    
    Args:
        query (str): User's question
        chunks (list[str]): Document chunks to search
    
    Returns:
        list[tuple]: List of (chunk_index, chunk_text, relevance_score) sorted by relevance
    """
    query_words = set(query.lower().split())
    scored_chunks = []
    
    for i, chunk in enumerate(chunks):
        chunk_words = set(chunk.lower().split())
        # Score based on word overlap
        overlap = len(query_words & chunk_words)
        if overlap > 0:
            scored_chunks.append((i, chunk, overlap))
    
    # Sort by relevance score (descending)
    scored_chunks.sort(key=lambda x: x[2], reverse=True)
    return scored_chunks[:3]  # Return top 3 chunks


def agent_pipeline(user_input):
    """
    Self-Verifying Agent Pipeline implementing the Trust Before Answer paradigm.
    Uses the built-in Webex knowledge base.
    
    Args:
        user_input (str): The user's question or query
    
    Returns:
        str: A verified answer, flagged answer, or safe refusal message
    """
    # Step 1: RETRIEVE — Search the Webex knowledge base
    context = retrieve_context(user_input)
    
    # Step 2: HARD GUARDRAIL — Knowledge sufficiency check
    if context is None:
        return ("🚫 Refusal: The system has detected insufficient knowledge "
                "to answer this query safely. The question falls outside the "
                "verified Webex knowledge base (source: help.webex.com).")
    
    # Step 3: GENERATE — Invoke the guardrail-constrained chain
    response = guardrail_chain.invoke({
        "context": context, 
        "question": user_input
    })
    
    # Step 4: SELF-VERIFY — Validate the generated answer against evidence
    verification_result = verification_chain.invoke({
        "evidence": context,
        "answer": response
    })
    
    # Step 5: PRESENT — Only show verified answers; flag unverified ones
    if "SUPPORTED" in verification_result.upper() and "NOT" not in verification_result.upper():
        return f"✅ [Verified] {response}"
    else:
        return (f"⚠️ [Unverified] The agent generated a response but self-verification "
                f"could not confirm it is fully supported by the evidence.\n"
                f"   Generated answer: {response}\n"
                f"   Verification status: {verification_result.strip()}\n"
                f"   Recommendation: Please verify this information at help.webex.com")


def agent_pipeline_with_document(user_input, uploaded_file):
    """
    Self-Verifying Agent Pipeline with Document Upload support.
    
    When a document is uploaded:
      1. The document is chunked using RecursiveCharacterTextSplitter
      2. Relevant chunks are retrieved based on the query
      3. The answer is generated from those chunks
      4. Self-verification validates the answer against the chunks
      5. The chunks used are displayed for transparency
    
    When no document is uploaded:
      Falls back to the standard Webex knowledge base pipeline.
    
    Args:
        user_input (str): The user's question
        uploaded_file: Gradio file upload object (path to temp file)
    
    Returns:
        tuple: (answer_text, chunks_display)
    """
    try:
        if not user_input or not user_input.strip():
            return "⚠️ Please enter a question.", ""
        
        # --- DOCUMENT MODE: Use uploaded file ---
        if uploaded_file is not None and str(uploaded_file).strip():
            # Step 1: CHUNK — Split document into pieces
            chunks = chunk_document(uploaded_file)
            
            if not chunks:
                return "🚫 Could not extract text from the uploaded document.", ""
            
            # Step 2: RETRIEVE — Find relevant chunks
            relevant_chunks = retrieve_from_chunks(user_input, chunks)
            
            if not relevant_chunks:
                # Format all chunks for display even if no match
                chunks_display = f"📄 Document split into {len(chunks)} chunks.\n"
                chunks_display += "⚠️ No relevant chunks found for your query.\n\n"
                for i, chunk in enumerate(chunks):
                    chunks_display += f"--- Chunk {i+1} ---\n{chunk[:200]}...\n\n"
                return ("🚫 Refusal: No relevant information found in the uploaded "
                        "document to answer this query safely."), chunks_display
            
            # Combine relevant chunks as context
            context = "\n\n".join([chunk_text for _, chunk_text, _ in relevant_chunks])
            
            # Step 3: GENERATE — Use guardrail chain with document context
            response = guardrail_chain.invoke({
                "context": context,
                "question": user_input
            })
            
            # Step 4: SELF-VERIFY — Validate against document evidence
            verification_result = verification_chain.invoke({
                "evidence": context,
                "answer": response
            })
            
            # Step 5: FORMAT CHUNKS DISPLAY — Show which chunks were used
            chunks_display = f"📄 Document split into {len(chunks)} total chunks.\n"
            chunks_display += f"🔍 {len(relevant_chunks)} relevant chunk(s) retrieved for verification:\n\n"
            
            for idx, (chunk_idx, chunk_text, score) in enumerate(relevant_chunks):
                chunks_display += f"━━━ Chunk {chunk_idx + 1} (relevance: {score} keyword matches) ━━━\n"
                chunks_display += f"{chunk_text}\n\n"
            
            # Add verification trace
            chunks_display += f"━━━ Verification Result ━━━\n"
            chunks_display += f"Verdict: {verification_result.strip()}\n"
            
            # Step 6: PRESENT — Verified or flagged
            if "SUPPORTED" in verification_result.upper() and "NOT" not in verification_result.upper():
                answer = f"✅ [Verified from Document] {response}"
            else:
                answer = (f"⚠️ [Unverified] The answer could not be fully confirmed "
                          f"against the document evidence.\n"
                          f"   Generated answer: {response}\n"
                          f"   Verification: {verification_result.strip()}")
            
            return answer, chunks_display
        
        # --- KNOWLEDGE BASE MODE: Use built-in Webex KB ---
        else:
            result = agent_pipeline(user_input)
            # Show which KB entry was matched
            context = retrieve_context(user_input)
            if context:
                chunks_display = (f"📚 Source: Webex Knowledge Base (help.webex.com)\n\n"
                                f"━━━ Retrieved Context ━━━\n{context}")
            else:
                chunks_display = "📚 No matching entry in Webex Knowledge Base."
            return result, chunks_display
    
    except Exception as e:
        return f"❌ Error: {str(e)}", f"An error occurred during processing: {str(e)}"


# --- Test the pipeline ---
print("Testing Self-Verifying Agent Pipeline:")
print("=" * 60)

# Test 1: KB mode
print("\n📝 Test 1: Webex Meetings query (KB mode)")
result1 = agent_pipeline("How do I join a meeting in Webex?")
print(f"   Response: {result1}")

# Test 2: Out-of-scope (KB mode)
print("\n📝 Test 2: Out-of-scope query (should REFUSE)")
result2 = agent_pipeline("What is the capital of France?")
print(f"   Response: {result2}")

# Test 3: Document chunking test
print("\n📝 Test 3: Document chunking (simulated)")
test_text = "This is a sample document. " * 50
test_chunks = text_splitter.split_text(test_text)
print(f"   Sample text ({len(test_text)} chars) → {len(test_chunks)} chunk(s)")

print("\n" + "=" * 60)
print("✅ Self-Verifying Agent tests complete!")
print("   Modes: Knowledge Base + Document Upload")
print("   Three layers: Hard Guardrail → Soft Guardrail → Self-Verification")

Testing Self-Verifying Agent Pipeline:

📝 Test 1: Webex Meetings query (KB mode)
   Response: ✅ [Verified] a single tap across any device — desktop, phone, or car — using the Move to Mobile QR code feature and Apple CarPlay integration

📝 Test 2: Out-of-scope query (should REFUSE)
   Response: ✅ [Verified] INSUFFICIENT_KNOWLEDGE: I cannot answer this.

📝 Test 3: Document chunking (simulated)
   Sample text (1350 chars) → 4 chunk(s)

✅ Self-Verifying Agent tests complete!
   Modes: Knowledge Base + Document Upload
   Three layers: Hard Guardrail → Soft Guardrail → Self-Verification


## Block 7: Gradio Web Interface (with Document Upload & Chunk Display)

This block creates an **interactive web UI** with **document upload** support that shows retrieved chunks during the verification process.

### UI Components:

| Component | Type | Purpose |
|-----------|------|--------|
| Title & Description | Markdown | Explains the self-verifying agent |
| Query Input | Textbox | Where users type their questions |
| Document Upload | File | Upload .txt, .pdf, .md, .csv to use as knowledge source |
| Agent Response | Textbox | Displays verified/unverified/refused response |
| Chunks Display | Markdown | **Shows retrieved chunks used for verification** |
| Status Legend | Markdown | Explains the ✅ ⚠️ 🚫 indicators |

### Two Modes of Operation:

| Mode | Trigger | Behavior |
|------|---------|----------|
| **Knowledge Base** | No file uploaded | Queries the built-in Webex KB |
| **Document Mode** | File uploaded | Chunks the document → retrieves relevant chunks → generates & verifies |

### Document Chunking Process:

```
Upload → Read Text → Split into Chunks (500 chars, 100 overlap) → Keyword Retrieval → Top 3 Chunks → Generate → Verify
```

### Supported File Types:
- `.txt` — Plain text files
- `.md` — Markdown files
- `.pdf` — PDF documents (text extraction)
- `.csv` — CSV data files

In [9]:
# Block 7: Gradio Web Interface — With Document Upload & Chunk Visualization
# Creates an interactive demo with document chunking transparency

# Fix for gradio_client schema bug (TypeError: argument of type 'bool' is not iterable)
import gradio_client.utils as _gc_utils

_original_get_type = _gc_utils.get_type

def _patched_get_type(schema):
    """Patched get_type to handle boolean schema values gracefully."""
    if not isinstance(schema, dict):
        return "any"
    return _original_get_type(schema)

_gc_utils.get_type = _patched_get_type

# Also patch the _json_schema_to_python_type to handle bool schema
_original_json_schema_to_python_type = _gc_utils._json_schema_to_python_type

def _patched_json_schema_to_python_type(schema, defs=None):
    """Patched to handle boolean additionalProperties."""
    if not isinstance(schema, dict):
        return "any"
    return _original_json_schema_to_python_type(schema, defs)

_gc_utils._json_schema_to_python_type = _patched_json_schema_to_python_type

print("✅ Applied gradio_client schema compatibility patch")


def launch_app():
    """
    Builds and launches the Gradio web interface with:
    - Document upload support (.txt, .pdf, .md, .csv)
    - Chunk visualization (shows which document chunks are used)
    - Self-verification status indicators
    - Two modes: Knowledge Base and Document mode
    """
    with gr.Blocks(title="Webex Self-Verifying AI Guardrail Agent") as demo:
        # Header section
        gr.Markdown("# 🛡️ Webex Self-Verifying AI Guardrail Agent")
        gr.Markdown(
            "A **Self-Verifying AI Agent** that validates its own answers against "
            "retrieved evidence before presenting them.\n\n"
            "**Three-Layer Protection:**\n"
            "1. 🔒 Hard Guardrail — Refuses when no relevant knowledge exists\n"
            "2. 📋 Soft Guardrail — Prompt constrains LLM to only use context\n"
            "3. 🔍 Self-Verification — Agent cross-checks its answer against evidence\n\n"
            "**Two Modes:** Use the built-in Webex KB, OR upload your own document!"
        )
        
        with gr.Row():
            with gr.Column(scale=2):
                # Input section
                input_text = gr.Textbox(
                    label="Your Query",
                    placeholder="Ask about Webex OR about your uploaded document...",
                    lines=2
                )
                
                # Document upload section
                file_upload = gr.File(
                    label="📄 Upload Document (optional: .txt, .pdf, .md, .csv)",
                    type="filepath"
                )
                gr.Markdown(
                    "*Upload a `.txt`, `.pdf`, `.md`, or `.csv` file to use as "
                    "your custom knowledge source. Leave empty to use Webex KB.*"
                )
                
                # Submit button
                submit_btn = gr.Button("🔍 Ask & Verify", variant="primary")
            
            with gr.Column(scale=2):
                # Output section
                output_text = gr.Textbox(
                    label="Agent Response (Self-Verified)",
                    lines=5,
                    interactive=False
                )
        
        # Chunks display section
        gr.Markdown("---")
        gr.Markdown("### 🧩 Retrieved Chunks & Verification Trace")
        chunks_output = gr.Textbox(
            label="Chunks & Verification Details",
            value="Submit a query to see retrieved chunks and verification details here.",
            lines=10,
            interactive=False
        )
        
        # Wire up the submit actions
        submit_btn.click(
            fn=agent_pipeline_with_document,
            inputs=[input_text, file_upload],
            outputs=[output_text, chunks_output]
        )
        
        # Also allow Enter key to submit
        input_text.submit(
            fn=agent_pipeline_with_document,
            inputs=[input_text, file_upload],
            outputs=[output_text, chunks_output]
        )
        
        # Status legend
        gr.Markdown(
            "---\n"
            "### 📊 Response Status Legend\n"
            "| Icon | Status | Meaning |\n"
            "|------|--------|--------|\n"
            "| ✅ | **Verified** | Answer generated AND confirmed against evidence |\n"
            "| ⚠️ | **Unverified** | Answer generated but NOT fully confirmed — use caution |\n"
            "| 🚫 | **Refused** | No relevant knowledge found — safe denial |\n\n"
            "| Mode | Description |\n"
            "|------|------------|\n"
            "| 📚 **KB Mode** | Uses built-in Webex knowledge base (no file uploaded) |\n"
            "| 📄 **Document Mode** | Chunks your uploaded file and retrieves from it |"
        )
        
        # Example queries
        gr.Markdown("### 💡 Try these examples:")
        gr.Examples(
            examples=[
                "How do I join a meeting in Webex?",
                "How do I share content in a Webex meeting?",
                "Tell me about Webex messaging and spaces",
                "How does noise removal work in Webex?",
                "Is Webex encrypted and secure?",
                "How do I create a Webex bot?",
                "What is the weather today?",
                "How do I use Microsoft Teams?"
            ],
            inputs=input_text
        )
    
    # Launch the app
    demo.launch(share=True)


print("✅ launch_app() function defined (with Document Upload & Chunk Display)")
print("   Features:")
print("   - 📄 Document upload (.txt, .pdf, .md, .csv)")
print("   - 🧩 Chunk visualization (shows retrieved chunks)")
print("   - 🔍 Verification trace (shows verification verdict)")
print("   - 📚 Dual mode: Webex KB + Custom Document")
print("   Run the next cell to start the Gradio interface")

✅ Applied gradio_client schema compatibility patch
✅ launch_app() function defined (with Document Upload & Chunk Display)
   Features:
   - 📄 Document upload (.txt, .pdf, .md, .csv)
   - 🧩 Chunk visualization (shows retrieved chunks)
   - 🔍 Verification trace (shows verification verdict)
   - 📚 Dual mode: Webex KB + Custom Document
   Run the next cell to start the Gradio interface


## Block 8: Launch the Application

Run this cell to start the Gradio web interface. A local URL will be displayed — click it to open the interactive demo in your browser.

**No API key needed!** Everything runs locally on your machine.

In [10]:
# Block 8: Launch the Gradio application
# This will start a local web server with the interactive UI

launch_app()

Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


---

## 📊 Detailed Project Summary

### Problem Statement

Large Language Models (LLMs) are prone to **hallucination** — generating plausible-sounding but factually incorrect information. This is particularly dangerous in enterprise support scenarios where users need accurate information about product features and capabilities. The "Trust Before Answer" system addresses this by implementing a **guardrail mechanism** combined with a **self-verification step** (inspired by [LangChain's RAG QA patterns](https://python.langchain.com/docs/use_cases/question_answering/)) that ensures every response is grounded in verified Webex documentation from help.webex.com.

### Knowledge Source

All knowledge in this system is sourced from **[help.webex.com](https://help.webex.com)**, covering:
- Webex Meetings (scheduling, joining, features)
- Webex Messaging & Spaces (collaboration, file sharing)
- Content Sharing (screen share, immersive share, attachments)
- Noise Removal & Virtual Backgrounds
- Security & End-to-End Encryption
- Webex REST API & Bot Development
- Webex Assistant (AI-powered meeting assistant)

### Solution Architecture (with Self-Verification)

```
┌─────────────────────────────────────────────────────────────┐
│                    USER INTERFACE (Gradio)                    │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│   User Query (e.g., "How do I join a Webex meeting?")       │
│       │                                                      │
│       ▼                                                      │
│   ┌─────────────────────┐                                   │
│   │  RETRIEVAL MODULE   │ ← Webex KB (help.webex.com)       │
│   │  retrieve_context() │                                   │
│   └─────────┬───────────┘                                   │
│             │                                                │
│             ▼                                                │
│   ┌─────────────────────┐                                   │
│   │  LAYER 1: HARD      │ context == None?                  │
│   │  GUARDRAIL          │──── YES ──→ SAFE REFUSAL 🚫       │
│   └─────────┬───────────┘                                   │
│             │ NO                                             │
│             ▼                                                │
│   ┌─────────────────────┐                                   │
│   │  LAYER 2: SOFT      │                                   │
│   │  GUARDRAIL          │ ← Ethical AI Guardrail Prompt     │
│   │  (Generation)       │                                   │
│   │  Flan-T5 (Local)    │                                   │
│   └─────────┬───────────┘                                   │
│             │                                                │
│             ▼                                                │
│   ┌─────────────────────────────┐                           │
│   │  LAYER 3: SELF-VERIFICATION │ ← NEW! (LangChain QA)    │
│   │  (Answer Validation)        │                           │
│   │  Checks: Is answer fully    │                           │
│   │  supported by evidence?     │                           │
│   └─────────┬───────────────────┘                           │
│             │                                                │
│        ┌────┴────┐                                          │
│        │         │                                          │
│    SUPPORTED  NOT_SUPPORTED                                  │
│        │         │                                          │
│        ▼         ▼                                          │
│   ✅ VERIFIED  ⚠️ FLAGGED                                   │
│    ANSWER      (with warning)                               │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

### Self-Verifying AI Agent Technique

Inspired by **LangChain's RAG question-answering patterns**, this project implements an internal verification step:

| Step | Action | Purpose |
|------|--------|---------|
| 1. Retrieve | Search knowledge base | Get relevant evidence |
| 2. Generate | LLM creates answer from context | Produce candidate response |
| 3. **Verify** | LLM checks answer against evidence | **Catch hallucinations/extrapolations** |
| 4. Present | Show verified or flagged result | Ensure user trust |

The verification chain asks: *"Is this answer fully supported by the evidence?"* — acting as an internal audit before any response reaches the user.

### Key Design Decisions

| Decision | Rationale |
|----------|----------|
| Local Flan-T5 model | No API key, no costs, no internet dependency |
| `do_sample=False` | Ensures deterministic, reproducible outputs |
| **Three-layer guardrails** | Hard (code) + Soft (prompt) + Verification (self-check) |
| **Self-verification chain** | Agent validates its own output before presenting (LangChain QA pattern) |
| Explicit refusal messages | User understands WHY the system can't answer |
| Context-only generation | LLM cannot use training data, only Webex documentation |
| help.webex.com as source | Authoritative, verified enterprise documentation |

### Ethical AI Principles Demonstrated

1. **Transparency**: The system clearly communicates when it cannot answer AND when an answer is unverified
2. **Truthfulness**: Answers are strictly grounded in verified Webex documentation
3. **Self-Accountability**: The agent audits its own responses before presenting them
4. **Safety**: The system errs on the side of refusal/flagging rather than fabrication
5. **Traceability**: Every answer can be traced back to help.webex.com content
6. **Accessibility**: Runs locally without paid APIs — democratized AI safety

### Models Used

| Model | Type | Size | Purpose |
|-------|------|------|--------|
| `google/flan-t5-base` | Seq2Seq (local) | 250M params | Guardrail-constrained generation + Self-verification |

### Techniques Incorporated

| Technique | Source | Implementation |
|-----------|--------|---------------|
| RAG (Retrieval-Augmented Generation) | LangChain Docs | `retrieve_context()` → guardrail chain |
| Ethical AI Guardrails | Prompt Engineering | Dual-layer (code + prompt) refusal mechanism |
| **Self-Verifying AI Agent** | [LangChain QA Patterns](https://python.langchain.com/docs/use_cases/question_answering/) | `verification_chain` validates answers against evidence |
| Chain Composition (LCEL) | LangChain Core | Pipe syntax for composable pipelines |

### Limitations & Future Improvements

| Limitation | Possible Improvement |
|-----------|---------------------|
| Keyword matching only | Use vector embeddings (FAISS/ChromaDB) for semantic search over full help.webex.com |
| Static knowledge base | Live crawling/indexing of help.webex.com articles |
| No confidence scoring | Add retrieval confidence threshold + verification confidence score |
| Same model for generation & verification | Use different model for verification (ensemble approach) |
| Flan-T5-base (250M) | Use larger model (flan-t5-large/xl) for better answers |
| Text only | Support multi-modal inputs (screenshots of Webex UI) |
| Binary verification | Add nuanced verification (fully supported / partially supported / not supported) |

---

*Built with LangChain · Self-Verifying AI Agent Pattern · Hugging Face Flan-T5 · Gradio · Knowledge from help.webex.com — No API key required!*